In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
crop = 'corn'
file_path = f'predictions/S2L2A_S1GRD_DEM_WEATHER_mask_11.csv'
df = pd.read_csv(file_path)  # Skip malformed lines
file = file_path.split('/')[-1].replace('.csv', '') 
df = df[df['YieldGT']>0]
df

In [ ]:
df['YieldGT'].iloc[0]

In [ ]:
crop = '' # Specify crop type: 'corn' or 'soybean' or leave empty for all
# Aggregate values by unique Filename (taking mean of predictions and ground truth)
df_agg = df.groupby('Filename').agg({
    'YieldGT': 'mean',
    'Prediction': 'mean'
}).reset_index()

if crop.lower() == 'corn':
    df_agg = df_agg[df_agg['Filename'].str.lower().str.contains('corn')]  # Filter for corn data only
elif crop.lower() == 'soybean':
    df_agg = df_agg[df_agg['Filename'].str.lower().str.contains('soybean')]  # Filter for soybean data only

print(f"Original records: {len(df_agg)}")
print(f"Unique files after aggregation: {len(df_agg)}")
df_agg.head()

In [ ]:
# Calculate metrics on aggregated data
from sklearn.metrics import mean_absolute_error, r2_score

# Calculate MAE
mae = mean_absolute_error(df_agg['YieldGT'], df_agg['Prediction'])

# Calculate MAPE
mape = np.mean(np.abs((df_agg['YieldGT'] - df_agg['Prediction']) / df_agg['YieldGT'])) * 100

# Calculate R-square
r2 = r2_score(df_agg['YieldGT'], df_agg['Prediction'])

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"R-squared (R²): {r2:.4f}")

In [ ]:
modal_code = 'S12WD' if 'WEATHER' in file_path else 'S12'
# Create publication-quality R-square plot
plt.figure(figsize=(8, 8), dpi=300)

# Set style for publication
sns.set_style("white")
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

# Separate corn and soybean data
df_corn = df_agg[df_agg['Filename'].str.lower().str.contains('corn')]
df_soybean = df_agg[df_agg['Filename'].str.lower().str.contains('soybean')]
if len(df_corn) == 0:
    res_name = 'Soybean'
else:
    res_name = 'Corn'
# Create scatter plots with different colors
plt.scatter(df_corn['YieldGT'], df_corn['Prediction'], alpha=0.5, s=30, 
            c='#2E86AB', edgecolors='none', label='Corn')
# plt.scatter(df_soybean['YieldGT'], df_soybean['Prediction'], alpha=0.5, s=30, 
#             c='#F77F00', edgecolors='none', label='Soybean')

# Add 1:1 line
min_val = min(df_agg['YieldGT'].min(), df_agg['Prediction'].min())
max_val = max(df_agg['YieldGT'].max(), df_agg['Prediction'].max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, 
         label='1:1 Line', alpha=0.7)

# Add regression line
z = np.polyfit(df_agg['YieldGT'], df_agg['Prediction'], 1)
p = np.poly1d(z)
plt.plot(df_agg['YieldGT'].sort_values(), p(df_agg['YieldGT'].sort_values()), 
         'r-', linewidth=2, label=f'Fit: y={z[0]:.3f}x+{z[1]:.3f}', alpha=0.7)

# Add metrics text box
textstr = f'R² = {r2:.4f}\nMAE = {mae:.4f}\nMAPE = {mape:.2f}%\nn = {len(df_agg)}'
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
plt.text(0.05, 0.95, textstr, transform=plt.gca().transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

# Labels and formatting
plt.xlabel('Observed Yield (Ground Truth)', fontsize=14, fontweight='bold')
plt.ylabel('Predicted Yield', fontsize=14, fontweight='bold')
plt.title('Model Performance: Predicted vs Observed Yield', fontsize=16, 
          fontweight='bold', pad=20)
# plt.xlim(0,0.7)
# plt.ylim(0,0.7)
# Legend
plt.legend(loc='lower right', frameon=True, fontsize=11, 
          edgecolor='gray', fancybox=False)

# Grid
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Equal aspect ratio
plt.axis('equal')
plt.tight_layout()

# Save figure
plt.savefig(f'figures/r_square_{modal_code}_{res_name}.png', dpi=300, bbox_inches='tight')
plt.savefig(f'figures/r_square_{modal_code}_{res_name}.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlots saved to 'figures/r_square_{modal_code}_{res_name}.png' and 'figures/r_square_{modal_code}_{res_name}.pdf'")

In [ ]:
import os
import glob
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Directory containing prediction CSV files
predictions_dir = 'predictions/'

# Get all CSV files in the predictions directory
csv_files = glob.glob(os.path.join(predictions_dir, 'S2L2A_*.csv'))

# Dictionary to store results
results = []

# Process each CSV file
for csv_file in csv_files:
    # Extract filename without path and extension
    base_filename = os.path.basename(csv_file)
    file_key = base_filename.replace('.csv', '')
    
    # Extract time from filename (last part after last underscore)
    try:
        time_value = int(file_key.split('_')[-1])
    except (ValueError, IndexError):
        print(f"Skipping {base_filename}: Cannot extract time value")
        continue
    
    print(f"Processing: {base_filename} (time={time_value})")
    
    # Read and process data
    df = pd.read_csv(csv_file)
    df = df[df['YieldGT'] > 0]
    
    # Aggregate by filename
    df_agg = df.groupby('Filename').agg({
        'YieldGT': 'mean',
        'Prediction': 'mean'
    }).reset_index()
    
    # Calculate metrics
    if len(df_agg) > 0:
        mae = mean_absolute_error(df_agg['YieldGT'], df_agg['Prediction'])
        mape = np.mean(np.abs((df_agg['YieldGT'] - df_agg['Prediction']) / df_agg['YieldGT'])) * 100
        r2 = r2_score(df_agg['YieldGT'], df_agg['Prediction'])
        
        results.append({
            'filename': file_key,
            'time': time_value,
            'r2': r2,
            'mae': mae,
            'mape': mape,
            'n_samples': len(df_agg)
        })
        
        print(f"  R²: {r2:.4f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%, n={len(df_agg)}")
    else:
        print(f"  Skipped: No valid data")

# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('time')
results_df.reset_index(drop=True, inplace=True)
results_df['mae'] = results_df['mae'] * 370  # Convert to Bu/Acre

print(f"\nProcessed {len(results_df)} files successfully")
results_df.head(15)

In [ ]:
results_df.head(15)

In [ ]:
# Create publication-quality plot of R² vs Time
plt.figure(figsize=(10, 6), dpi=300)

# Set style for publication
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

# Plot R² vs time
plt.plot(results_df['time'], results_df['r2'], marker='o', linewidth=2, 
         markersize=8, color='#2E86AB', label='R² Score')

# Fill area under curve for visual appeal
plt.fill_between(results_df['time'], results_df['r2'], alpha=0.3, color='#2E86AB')

# Labels and formatting
plt.xlabel('Time (Last Underscore Value)', fontsize=14, fontweight='bold')
plt.ylabel('R² Score', fontsize=14, fontweight='bold')
plt.title('Model Performance Over Time', fontsize=16, fontweight='bold', pad=20)

# Add horizontal line at R²=1 for reference
plt.axhline(y=1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Perfect Fit')

# Grid
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Legend
plt.legend(loc='best', frameon=True, fontsize=11, edgecolor='gray')

# Set y-axis limits to show full range
plt.ylim([0, 1.05])

plt.tight_layout()

# Save figure
plt.savefig('figures/r2_vs_time.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/r2_vs_time.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to 'figures/r2_vs_time.png' and 'figures/r2_vs_time.pdf'")
print(f"\nSummary Statistics:")
print(f"Mean R²: {results_df['r2'].mean():.4f}")
print(f"Max R²: {results_df['r2'].max():.4f} at time {results_df.loc[results_df['r2'].idxmax(), 'time']}")
print(f"Min R²: {results_df['r2'].min():.4f} at time {results_df.loc[results_df['r2'].idxmin(), 'time']}")

In [ ]:
import os
import glob
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Directory containing prediction CSV files
predictions_dir = 'predictions/'

# Get all CSV files in the predictions directory
csv_files = glob.glob(os.path.join(predictions_dir, '*_DEM_*.csv'))

# Dictionary to store results
results = []

# Process each CSV file
for csv_file in csv_files:
    # Extract filename without path and extension
    base_filename = os.path.basename(csv_file)
    file_key = base_filename.replace('.csv', '')
    
    # Extract time from filename (last part after last underscore)
    try:
        time_value = int(file_key.split('_')[-1])
    except (ValueError, IndexError):
        print(f"Skipping {base_filename}: Cannot extract time value")
        continue
    
    print(f"Processing: {base_filename} (time={time_value})")
    
    # Read and process data
    df = pd.read_csv(csv_file)
    df = df[df['YieldGT'] > 0]
    
    # Aggregate by filename
    df_agg = df.groupby('Filename').agg({
        'YieldGT': 'mean',
        'Prediction': 'mean'
    }).reset_index()
    
    # Calculate metrics
    if len(df_agg) > 0:
        mae = mean_absolute_error(df_agg['YieldGT'], df_agg['Prediction'])
        mape = np.mean(np.abs((df_agg['YieldGT'] - df_agg['Prediction']) / df_agg['YieldGT'])) * 100
        r2 = r2_score(df_agg['YieldGT'], df_agg['Prediction'])
        
        results.append({
            'filename': file_key,
            'time': time_value,
            'r2': r2,
            'mae': mae,
            'mape': mape,
            'n_samples': len(df_agg)
        })
        
        print(f"  R²: {r2:.4f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%, n={len(df_agg)}")
    else:
        print(f"  Skipped: No valid data")

# Convert results to DataFrame
results_df_wd = pd.DataFrame(results)
results_df_wd = results_df_wd.sort_values('time')
results_df_wd.reset_index(drop=True, inplace=True)
results_df_wd['mae'] = results_df_wd['mae'] * 370  # Convert to Bu/Acre

print(f"\nProcessed {len(results_df_wd)} files successfully")
results_df_wd.head(15)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

_code = "s12wd"  # Specify the code used in filenames
# Set seaborn style for publication
sns.set(style="whitegrid", context="talk", font_scale=1.3)

# Load results
df = pd.read_csv(f"classical_ml_results_{_code}.csv")
df["biweek_num"] = df["biweek"].str.extract(r"(\d+)").astype(int)
df['test_mae'] = df['test_mae'] * 370
# Prepare R² data
r2_long = df.melt(
    id_vars=["biweek_num", "model"],
    value_vars=["test_r2"],
    var_name="split",
    value_name="R²"
)
r2_long["split"] = r2_long["split"].str.replace("_r2", "").str.capitalize()

# Plot R²
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=r2_long,
    x="biweek_num",
    y="R²",
    hue="model",
    marker="o",
    palette="Set1"
)
plt.title("Model $R^2$ Across Biweeks", fontsize=20, pad=15)
plt.ylim(0.0, 1.0)
plt.xlabel("Biweek", fontsize=16)
plt.ylabel("$R^2$ Score", fontsize=16)
plt.xticks(df["biweek_num"].unique())
plt.legend(title="Model", loc="best")
plt.tight_layout()
plt.savefig("classical_ml_r2_biweek.pdf", bbox_inches="tight", dpi=300)
plt.show()

# Prepare MAE data
mae_long = df.melt(
    id_vars=["biweek_num", "model"],
    value_vars=["test_mae"],
    var_name="split",
    value_name="MAE"
)
mae_long["split"] = mae_long["split"].str.replace("_mae", "").str.capitalize()

# Plot MAE
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=mae_long,
    x="biweek_num",
    y="MAE",
    hue="model",
    marker="o",
    palette="Set1"
)
plt.title("Model MAE Across Biweeks", fontsize=20, pad=15)
plt.xlabel("Biweek", fontsize=16)
plt.ylabel("MAE (Bu/Acre)", fontsize=16)
plt.xticks(df["biweek_num"].unique())
plt.ylim(20, 65)
plt.legend(title="Model", loc="best")
plt.tight_layout()
plt.savefig("classical_ml_mae_biweek.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Load classical ML results for comparison
_code = "s12wd"  # Specify the code used in filenames
df_classical = pd.read_csv(f"classical_ml_results_{_code}.csv")
df_classical["biweek_num"] = df_classical["biweek"].str.extract(r"(\d+)").astype(int)
df_classical['test_mae'] = df_classical['test_mae'] * 370  # Convert to Bu/Acre

# Get best model results from classical ML (e.g., XgBoost)
df_classical_rf = df_classical[df_classical['model'] == 'XGBoost'].reset_index(drop=True)
df_classical_rf = df_classical_rf.sort_values('biweek_num').reset_index(drop=True)
df_classical_plsr = df_classical[df_classical['model'] == 'PLSR'].reset_index(drop=True)
df_classical_plsr = df_classical_plsr.sort_values('biweek_num').reset_index(drop=True)
# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=300)

# Set style for publication
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

# Plot 1: R² comparison
ax1.plot(results_df_wd['time'], results_df_wd['r2'], marker='o', linewidth=2.5, 
         markersize=8, color='#2E86AB', label='TerraMind', alpha=0.8)
ax1.plot(df_classical_rf['biweek_num'], df_classical_rf['test_r2'], 
         marker='s', linewidth=2.5, markersize=8, color='#A23B72', 
         label='XgBoost', alpha=0.8)
ax1.plot(df_classical_plsr['biweek_num'], df_classical_plsr['test_r2'], 
         marker='^', linewidth=2.5, markersize=8, color='#D95F02', 
         label='PLSR', alpha=0.8)

ax1.set_xlabel('Biweek', fontsize=14, fontweight='bold')
ax1.set_ylabel('R² Score', fontsize=14, fontweight='bold')
ax1.set_title('R² Score Comparison: TerraMind (S12WD) vs Classical ML', 
              fontsize=14, fontweight='bold', pad=15)
ax1.axhline(y=1.0, color='gray', linestyle='--', linewidth=1, alpha=0.3)
ax1.set_ylim([0, 1.05])
ax1.legend(loc='best', frameon=True, fontsize=11, edgecolor='gray')
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Plot 2: MAE comparison
ax2.plot(results_df_wd['time'], results_df_wd['mae'], marker='o', linewidth=2.5, 
         markersize=8, color='#2E86AB', label='TerraMind', alpha=0.8)
ax2.plot(df_classical_rf['biweek_num'], df_classical_rf['test_mae'], 
         marker='s', linewidth=2.5, markersize=8, color='#A23B72', 
         label='XgBoost', alpha=0.8)
ax2.plot(df_classical_plsr['biweek_num'], df_classical_plsr['test_mae'], 
         marker='^', linewidth=2.5, markersize=8, color='#D95F02', 
         label='PLSR', alpha=0.8)

ax2.set_xlabel('Biweek', fontsize=14, fontweight='bold')
ax2.set_ylabel('MAE (Bu/Acre)', fontsize=14, fontweight='bold')
ax2.set_title('MAE Comparison: TerraMind (S12WD) vs Classical ML', 
              fontsize=14, fontweight='bold', pad=15)
ax2.legend(loc='best', frameon=True, fontsize=11, edgecolor='gray')
ax2.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

plt.tight_layout()

# Save figure
plt.savefig('figures/dlS12WD_vs_classical_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/dlS12WD_vs_classical_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("TerraMind SUMMARY:")
print("="*60)
print(f"Mean R²: {results_df['r2'].mean():.4f} (±{results_df['r2'].std():.4f})")
print(f"Mean MAE: {results_df['mae'].mean():.4f} (±{results_df['mae'].std():.4f})")
print(f"Best R²: {results_df['r2'].max():.4f} at biweek {results_df.loc[results_df['r2'].idxmax(), 'time']}")
print(f"Best MAE: {results_df['mae'].min():.4f} at biweek {results_df.loc[results_df['mae'].idxmin(), 'time']}")

print("\n" + "="*60)
print("XGBoost SUMMARY:")
print("="*60)
print(f"Mean R²: {df_classical_rf['test_r2'].mean():.4f} (±{df_classical_rf['test_r2'].std():.4f})")
print(f"Mean MAE: {df_classical_rf['test_mae'].mean():.4f} (±{df_classical_rf['test_mae'].std():.4f})")
print(f"Best R²: {df_classical_rf['test_r2'].max():.4f} at biweek {df_classical_rf.loc[df_classical_rf['test_r2'].idxmax(), 'biweek_num']}")
print(f"Best MAE: {df_classical_rf['test_mae'].min():.4f} at biweek {df_classical_rf.loc[df_classical_rf['test_mae'].idxmin(), 'biweek_num']}")
print("="*60)

print("\n" + "="*60)
print("PLSR SUMMARY:")
print("="*60)
print(f"Mean R²: {df_classical_plsr['test_r2'].mean():.4f} (±{df_classical_plsr['test_r2'].std():.4f})")
print(f"Mean MAE: {df_classical_plsr['test_mae'].mean():.4f} (±{df_classical_plsr['test_mae'].std():.4f})")
print(f"Best R²: {df_classical_plsr['test_r2'].max():.4f} at biweek {df_classical_plsr.loc[df_classical_plsr['test_r2'].idxmax(), 'biweek_num']}")
print(f"Best MAE: {df_classical_plsr['test_mae'].min():.4f} at biweek {df_classical_plsr.loc[df_classical_plsr['test_mae'].idxmin(), 'biweek_num']}")
print("="*60)